In [26]:
# functions
import numpy as np
from scipy.stats import entropy
def jsd_multiple(distributions, weights=None, base=2):
    """
    Generalized Jensen-Shannon divergence across N >= 2 distributions.

    distributions: array of shape (n_distributions, n_categories),
                    each row summing to 1
    weights: optional weights for each distribution (e.g. relative sample
              sizes); defaults to equal weighting
    base: log base for entropy (2 = bits, np.e = nats)

    JSD = H(weighted mean distribution) - weighted_mean(H(each distribution))
    """
    distributions = np.asarray(distributions, dtype=float)
    n = distributions.shape[0]

    if weights is None:
        weights = np.full(n, 1.0 / n)
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights / weights.sum()  # normalize just in case

    mean_dist = np.average(distributions, axis=0, weights=weights)

    entropy_of_mean = entropy(mean_dist, base=base)
    mean_of_entropies = np.average(
        [entropy(d, base=base) for d in distributions], weights=weights
    )

    return entropy_of_mean - mean_of_entropies
def format_data(file_name,modifiers):
    df = read_csv(file_name)
    df = df[df.columns[df.columns.isin(["doc_id", "predicate", "attitude", "relationship", "modifier_response_list"])]]

    # take out these columns: doc_id,predicate,attitude,relationship,modifier_response_list where modifier list is a string like none; too; very; pretty; really; quite 
    df["is_negated"] = df.apply(lambda x: (x["attitude"], x["predicate"]) in NEGATED_SET, axis=1)
    # make each modifier in the modifier response list a separate row, so if there are three modifiers in the list, there will be three rows with the same doc_id, predicate, attitude, relationship, and is_negated value
    df = df.assign(modifier=df["modifier_response_list"].str.split(";")).explode("modifier").reset_index(drop=True)
    df["modifier"] = df["modifier"].str.strip()
    df.drop(columns=["modifier_response_list"], inplace=True)
    # filter rows where modifier is not in EN_MODIFIERS
    return df[df["modifier"].isin(modifiers)]
def filter_axis(axis,MODIFIERS,df):
    modifier_distribution = df.groupby([axis, "modifier"]).size().reset_index(name="count")
    modifier_distribution = modifier_distribution.pivot(index=axis, columns="modifier", values="count")
    modifier_distribution = modifier_distribution.reindex(columns=MODIFIERS, fill_value=0)  # ensure ALL modifiers present, even unobserved ones
    modifier_distribution.fillna(0, inplace=True)  # fill NaN with 0 for unobserved modifiers
    return modifier_distribution.div(modifier_distribution.sum(axis=1), axis=0)

In [27]:
JP_MODIFIERS = ["あまり", "いまいち", "かなり", "すごく", "そこまで", "それほど", "そんなに", "たいして", "だいぶ", "ちっとも", "ちょっと", "とても", "なかなか", "（なし）", "ひどく", "まあまあ", "マジで", "めっちゃ", "やや", "結構", "若干", "少し", "少しも", "全然", "相当", "超", "非常に", "微妙に", "普通に", "本当に", "全く"]
EN_MODIFIERS = ["very", "really", "that", "quite", "(none)", "too", "slightly", "somewhat", "pretty", "a little bit", "at all", "kind of", "completely", "kinda", "a bit", "semi", "a tad", "incredibly", "sorta", "totally", "a little", "amazingly", "extremely", "moderately","mildly",  "clearly", "damn", "majorly",  "absolutely", "exceptionally","so"]
NEGATED_SET = {
    ("non-committal", "面白い"),
    ("non-committal", "美味しい"),
    ("non-committal", "綺麗"),
    ("warning",       "面白い"),
    ("warning",       "美味しい"),
    ("warning",       "綺麗"),
    ("annoyed",       "面白い"),
    ("annoyed",       "美味しい"),
    ("annoyed",       "綺麗"),
    ("encouraging",   "遅れている"),
    ("encouraging",   "寒い"),
    ("encouraging",   "汚い"),
    ("acknowledge",   "遅れている"),
    ("acknowledge",   "寒い"),
    ("acknowledge",   "汚い"),
}

In [34]:
# read in csv files and create a file that 
from pandas import read_csv
en_df = format_data("/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPolitenessOfModifiers/data_analysis/EN_trials.csv", EN_MODIFIERS)
jp_df = format_data("/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPolitenessOfModifiers/data_analysis/JP_trials.csv", JP_MODIFIERS)
en_modifier_dist = filter_axis("relationship", EN_MODIFIERS,en_df)
jp_modifier_dist = filter_axis("relationship", JP_MODIFIERS,jp_df)
print("JSD across relationships in English:", jsd_multiple(en_modifier_dist.values))
print("JSD across relationships in Japanese:", jsd_multiple(jp_modifier_dist.values))
en_modifier_dist = filter_axis("attitude", EN_MODIFIERS,en_df)
jp_modifier_dist = filter_axis("attitude", JP_MODIFIERS,jp_df)
print("JSD across attitudes in English:", jsd_multiple(en_modifier_dist.values))
print("JSD across attitudes in Japanese:", jsd_multiple(jp_modifier_dist.values))
# same thing but when en_df and jp_df are filtered to only include rows where is_negated is False
en_df_pos = en_df[en_df["is_negated"] == False]
jp_df_pos = jp_df[jp_df["is_negated"] == False]
en_modifier_dist = filter_axis("relationship", EN_MODIFIERS,en_df_pos)
jp_modifier_dist = filter_axis("relationship", JP_MODIFIERS,jp_df_pos)
print("JSD across relationships in English (negated filtered):", jsd_multiple(en_modifier_dist.values))
print("JSD across relationships in Japanese (negated filtered):", jsd_multiple(jp_modifier_dist.values))
en_modifier_dist = filter_axis("attitude", EN_MODIFIERS,en_df_pos)
jp_modifier_dist = filter_axis("attitude", JP_MODIFIERS,jp_df_pos)
print("JSD across attitudes in English (negated filtered):", jsd_multiple(en_modifier_dist.values))
print("JSD across attitudes in Japanese (negated filtered):", jsd_multiple(jp_modifier_dist.values))

# same thing but when en_df and jp_df are filtered to only include rows where is_negated is True
en_df_neg = en_df[en_df["is_negated"] == True]
jp_df_neg = jp_df[jp_df["is_negated"] == True]
en_modifier_dist = filter_axis("relationship", EN_MODIFIERS,en_df_neg)
jp_modifier_dist = filter_axis("relationship", JP_MODIFIERS,jp_df_neg)
print("JSD across relationships in English (negated only):", jsd_multiple(en_modifier_dist.values))
print("JSD across relationships in Japanese (negated only):", jsd_multiple(jp_modifier_dist.values))
en_modifier_dist = filter_axis("attitude", EN_MODIFIERS,en_df_neg)
jp_modifier_dist = filter_axis("attitude", JP_MODIFIERS,jp_df_neg)
print("JSD across attitudes in English (negated only):", jsd_multiple(en_modifier_dist.values))
print("JSD across attitudes in Japanese (negated only):", jsd_multiple(jp_modifier_dist.values))

JSD across relationships in English: 0.10737991936777336
JSD across relationships in Japanese: 0.1279319746334675
JSD across attitudes in English: 0.3687263110239942
JSD across attitudes in Japanese: 0.8528222042501348
JSD across relationships in English (negated filtered): 0.1564974072147236
JSD across relationships in Japanese (negated filtered): 0.14199558535115298
JSD across attitudes in English (negated filtered): 0.483795724293822
JSD across attitudes in Japanese (negated filtered): 0.8322071281871453
JSD across relationships in English (negated only): 0.13731152480656306
JSD across relationships in Japanese (negated only): 0.2114044253368892
JSD across attitudes in English (negated only): 0.2622441747348736
JSD across attitudes in Japanese (negated only): 0.7625474094571727


In [ ]:
"""
Bare-minimum bootstrap CI for JSD(attitudes), English vs Japanese.
Paste into the existing notebook; uses its jsd_multiple() and *_df / *_MODIFIERS.

ONE FIX FIRST, in format_data():
    df["modifier"] = df["modifier"].str.strip()
before the .isin(modifiers) filter. Splitting on ";" leaves a leading space on every
modifier after the first, so "; "-separated cells silently lose all but the first.
"""

import numpy as np


def boot_jsd(df, MODIFIERS, cell_col, axis="attitude", n_boot=2000, seed=0, verbose=True):
    """Resample cells (= participants, one stimulus each) with replacement within each
    attitude, keeping the per-attitude count fixed. Returns (observed, bootstrap array)."""
    # --- drop rows with a missing key -------------------------------------
    # A NaN in `axis` or `cell_col` is what produces
    #   TypeError: '<' not supported between instances of 'float' and 'str'
    # from sorted(): pandas reads blank cells as float NaN, which cannot be
    # compared with the string labels. These rows cannot be placed in a cell or a
    # level, so they are dropped rather than coerced.
    n0 = len(df)
    df = df[df[axis].notna() & df[cell_col].notna() & df["modifier"].notna()].copy()
    if verbose and len(df) < n0:
        print(f"  dropped {n0 - len(df)} rows with missing {axis}/{cell_col}/modifier")
    assert len(df), f"no rows left after dropping missing {axis}/{cell_col}"

    df[axis] = df[axis].astype(str)
    df[cell_col] = df[cell_col].astype(str)

    mix = {m: i for i, m in enumerate(MODIFIERS)}
    levels = sorted(df[axis].unique())
    if verbose:
        print(f"  {len(levels)} {axis} levels: {levels}")
    aix = {a: i for i, a in enumerate(levels)}

    # one row per cell
    g = df.groupby([cell_col, axis, "modifier"]).size().reset_index(name="n")
    keys = g[[cell_col, axis]].drop_duplicates().reset_index(drop=True)
    if not keys[cell_col].is_unique:
        bad = keys[cell_col].value_counts()
        bad = bad[bad > 1]
        raise AssertionError(
            f"{len(bad)} values of '{cell_col}' appear under more than one {axis} "
            f"(e.g. {list(bad.index[:3])}), so it is not one cell per participant.\n"
            f"If participants really did see several {axis} levels, resample participants "
            f"as whole units by passing a participant-id column; if this is a composite-key "
            f"problem, build one, e.g.\n"
            f'    df["cell"] = df["{cell_col}"].astype(str) + "|" + df["{axis}"].astype(str)')
    kix = {k: i for i, k in enumerate(keys[cell_col])}

    C = np.zeros((len(keys), len(MODIFIERS)))
    np.add.at(C, (g[cell_col].map(kix).to_numpy(),
                  g["modifier"].map(mix).to_numpy()), g["n"].to_numpy())
    A = keys[axis].map(aix).to_numpy()
    groups = [np.flatnonzero(A == k) for k in range(len(levels))]

    def jsd_of(pick):
        Mat = np.zeros((len(levels), len(MODIFIERS)))
        np.add.at(Mat, A[pick], C[pick])
        t = Mat.sum(axis=1)
        k = t > 0
        return jsd_multiple(Mat[k] / t[k, None])

    rng = np.random.default_rng(seed)
    obs = jsd_of(np.arange(len(C)))
    boot = np.array([
        jsd_of(np.concatenate([gr[rng.integers(0, len(gr), len(gr))] for gr in groups]))
        for _ in range(n_boot)
    ])
    return obs, boot


def diagnose(df, cell_col=None, axis="attitude"):
    """Run this first if anything errors. Shows dtypes, missing values, and whether the
    candidate cell column is one-cell-per-participant."""
    print(f"{len(df)} rows")
    for c in [axis, "relationship", "modifier", cell_col]:
        if c is None or c not in df.columns:
            if c is not None:
                print(f"  {c:<20} MISSING from the dataframe")
            continue
        col = df[c]
        n_na = int(col.isna().sum())
        types = sorted({type(v).__name__ for v in col.dropna().unique()[:200]})
        print(f"  {c:<20} {col.nunique()} unique, {n_na} missing, types {types}")
        if n_na:
            print(f"  {'':<20} ^ these rows will be dropped; NaN is what breaks sorted()")
    if cell_col and cell_col in df.columns and axis in df.columns:
        sub = df[df[cell_col].notna() & df[axis].notna()]
        per = sub.groupby(cell_col)[axis].nunique()
        print(f"  {axis} levels per {cell_col}: min {per.min()}, max {per.max()}"
              + ("   -> one cell per participant, good"
                 if per.max() == 1 else
                 f"   -> spans {per.max()}; see the AssertionError message"))


def compare(en_df, jp_df, EN_MODIFIERS, JP_MODIFIERS, cell_col,
            axis="attitude", n_boot=2000, seed=0):
    print("EN:"); e_obs, e_boot = boot_jsd(en_df, EN_MODIFIERS, cell_col, axis, n_boot, seed)
    print("JP:"); j_obs, j_boot = boot_jsd(jp_df, JP_MODIFIERS, cell_col, axis, n_boot, seed)
    print()
    d = j_boot - e_boot
    lo, hi = np.percentile(d, [2.5, 97.5])
    frac = (d <= 0).mean()
    p = max(min(1.0, 2 * min(frac, 1 - frac)), 1 / (n_boot + 1))
    print(f"JSD across {axis}:  EN {e_obs:.4f}   JP {j_obs:.4f}")
    print(f"Delta (JP - EN) = {j_obs - e_obs:+.4f} bits")
    print(f"95% CI          = [{lo:+.4f}, {hi:+.4f}]")
    print(f"p               {'< ' + format(1/(n_boot+1), '.2g') if frac in (0.0, 1.0) else '= ' + format(p, '.4f')}"
          f"  (two-sided)")
    return dict(en=e_obs, jp=j_obs, delta=j_obs - e_obs, ci=(lo, hi), p=p, d=d)
en_df["cell"] = en_df["doc_id"].astype(str) + "|" + en_df["attitude"].astype(str)
jp_df["cell"] = jp_df["doc_id"].astype(str) + "|" + jp_df["attitude"].astype(str)
compare(en_df, jp_df, EN_MODIFIERS, JP_MODIFIERS, cell_col="cell", axis="attitude", n_boot=2000, seed=0)

[English]
  dropped 2 rows with missing attitude/doc_id/modifier
  6 attitude levels: ['acknowledge', 'annoyed', 'encouraging', 'neutral', 'non-committal', 'warning']


AssertionError: 10 values of 'doc_id' appear under more than one attitude (e.g. ['5a4bf09053a4560001ba0ebf', '65e975039c7865e835a2936c', '677650da17489eb0f07dec0d']). Use a column that is unique per participant, or build one: df["cell"] = df["doc_id"].astype(str) + "|" + df["attitude"].astype(str)

In [ ]:
# =============================================================================
# Is JSD across attitudes greater than chance, within one language?
#   H0: all attitude levels share the same underlying modifier distribution
#
# Paste this whole thing into one notebook cell. Self-contained except for
# jsd_multiple(), which your notebook already defines.
#
#
# Usage, after running the cell:
#   r_en = test_language(en_df, EN_MODIFIERS, "English",  cell_col="participant_id")
#   r_jp = test_language(jp_df, JP_MODIFIERS, "Japanese", cell_col="participant_id")
# =============================================================================

import numpy as np


def build_cells(df, MODIFIERS, cell_col, axis="attitude", verbose=True):
    """-> (C, A, levels). C[c, m] = times cell c produced modifier m;
    A[c] = which axis level cell c belongs to. One cell = one participant."""
    n0 = len(df)
    df = df[df[axis].notna() & df[cell_col].notna() & df["modifier"].notna()].copy()
    if verbose and len(df) < n0:
        print(f"  dropped {n0 - len(df)} rows with missing {axis}/{cell_col}/modifier")
    assert len(df), f"no rows left after dropping missing {axis}/{cell_col}"
    df[axis] = df[axis].astype(str)
    df[cell_col] = df[cell_col].astype(str)

    mix = {m: i for i, m in enumerate(MODIFIERS)}
    levels = sorted(df[axis].unique())
    if verbose:
        print(f"  {len(levels)} {axis} levels: {levels}")
    aix = {a: i for i, a in enumerate(levels)}

    g = df.groupby([cell_col, axis, "modifier"]).size().reset_index(name="n")
    keys = g[[cell_col, axis]].drop_duplicates().reset_index(drop=True)
    if not keys[cell_col].is_unique:
        bad = keys[cell_col].value_counts()
        bad = bad[bad > 1]
        raise AssertionError(
            f"{len(bad)} values of '{cell_col}' appear under more than one {axis} "
            f"(e.g. {list(bad.index[:3])}). Use a column that is unique per participant, "
            f'or build one: df["cell"] = df["{cell_col}"].astype(str) + "|" + '
            f'df["{axis}"].astype(str)')
    kix = {k: i for i, k in enumerate(keys[cell_col])}

    C = np.zeros((len(keys), len(MODIFIERS)))
    np.add.at(C, (g[cell_col].map(kix).to_numpy(),
                  g["modifier"].map(mix).to_numpy()), g["n"].to_numpy())
    return C, keys[axis].map(aix).to_numpy(), levels


def _jsd(C, A, K):
    Mat = np.zeros((K, C.shape[1]))
    np.add.at(Mat, A, C)
    t = Mat.sum(axis=1)
    k = t > 0
    return jsd_multiple(Mat[k] / t[k, None]) if k.sum() > 1 else np.nan


def test_language(df, MODIFIERS, label, cell_col, axis="attitude",
                  n_perm=5000, n_boot=1000, seed=0, alpha=0.05, verbose=True):
    print(f"[{label}]")
    C, A, levels = build_cells(df, MODIFIERS, cell_col, axis, verbose)
    K, N = len(levels), C.shape[0]
    rng = np.random.default_rng(seed)

    obs = _jsd(C, A, K)

    # permutation: reassign which cell carries which attitude label.
    # permuting A preserves the number of cells per attitude automatically.
    null = np.array([_jsd(C, rng.permutation(A), K) for _ in range(n_perm)])
    floor = float(np.nanmean(null))
    effect = obs - floor
    n_ge = int((null >= obs).sum())
    p = (n_ge + 1) / (n_perm + 1)
    p_floor = 1.0 / (n_perm + 1)

    # bootstrap: WIDTH only, recentred on the corrected effect. Resampling cells
    # with replacement inflates a non-negative divergence, so its location is
    # biased upward even though its spread is right.
    raw = np.array([_jsd(C[i], A[i], K)
                    for i in (rng.integers(0, N, N) for _ in range(n_boot))])
    mid = float(np.nanmedian(raw))
    lo_off, hi_off = np.nanpercentile(raw, [100 * alpha / 2, 100 * (1 - alpha / 2)]) - mid
    lo, hi = effect + lo_off, effect + hi_off
    sd = np.nanstd(null, ddof=1)

    print(f"  observed JSD        {obs:.4f} bits")
    print(f"  chance floor        {floor:.4f} bits   <- what identical attitudes give")
    print(f"  effect (obs-floor)  {effect:.4f} bits")
    print(f"  95% CI              [{lo:+.4f}, {hi:+.4f}]")
    print(f"  p                   "
          f"{'< ' + format(p_floor, '.2g') if n_ge == 0 else '= ' + format(p, '.4f')}"
          f"   ({n_ge}/{n_perm} permutations reached it)")
    print(f"  null: mean {floor:.4f}, sd {sd:.4f}, max {np.nanmax(null):.4f};  "
          f"observed is {(obs - floor) / sd:.1f} null SDs out")
    print(f"  -> {'attitudes DO differ' if (n_ge == 0 or p < alpha) else 'no detectable difference'}"
          f" (ceiling log2({K}) = {np.log2(K):.3f} bits)\n")

    return dict(label=label, obs=obs, floor=floor, effect=effect, ci=(lo, hi),
                p=p, p_censored=(n_ge == 0), null=null, boots=raw)
en_df["cell"] = en_df["doc_id"].astype(str) + "|" + en_df["attitude"].astype(str)
jp_df["cell"] = jp_df["doc_id"].astype(str) + "|" + jp_df["attitude"].astype(str)
test_language(en_df, EN_MODIFIERS, "English", cell_col="cell")
test_language(jp_df, JP_MODIFIERS, "Japanese", cell_col="cell")
# do the same for relationship instead of attitude
en_df["cell"] = en_df["doc_id"].astype(str) + "|" + en_df["relationship"].astype(str)
jp_df["cell"] = jp_df["doc_id"].astype(str) + "|" + jp_df["relationship"].astype(str)
test_language(en_df, EN_MODIFIERS, "English", cell_col="cell", axis="relationship")
test_language(jp_df, JP_MODIFIERS, "Japanese", cell_col="cell", axis="relationship")

[English]
  dropped 2 rows with missing attitude/cell/modifier
  6 attitude levels: ['acknowledge', 'annoyed', 'encouraging', 'neutral', 'non-committal', 'warning']
  observed JSD        0.3687 bits
  chance floor        0.3246 bits   <- what identical attitudes give
  effect (obs-floor)  0.0441 bits
  95% CI              [-0.0688, +0.1857]
  p                   = 0.0692   (345/5000 permutations reached it)
  null: mean 0.3246, sd 0.0290, max 0.4889;  observed is 1.5 null SDs out
  -> no detectable difference (ceiling log2(6) = 2.585 bits)

[Japanese]
  6 attitude levels: ['acknowledge', 'annoyed', 'encouraging', 'neutral', 'non-committal', 'warning']
  observed JSD        0.8528 bits
  chance floor        0.4171 bits   <- what identical attitudes give
  effect (obs-floor)  0.4357 bits
  95% CI              [+0.3124, +0.5794]
  p                   < 0.0002   (0/5000 permutations reached it)
  null: mean 0.4171, sd 0.0445, max 0.5873;  observed is 9.8 null SDs out
  -> attitudes DO diff

{'label': 'Japanese',
 'obs': np.float64(0.1279319746334675),
 'floor': 0.18251388725982548,
 'effect': np.float64(-0.054581912626357976),
 'ci': (np.float64(-0.1395067522225932), np.float64(0.05288435177627304)),
 'p': 0.9758048390321936,
 'p_censored': False,
 'null': array([0.17001787, 0.12524943, 0.16704155, ..., 0.13751251, 0.18682869,
        0.12507186]),
 'boots': array([0.34398779, 0.28869925, 0.28725762, 0.34742913, 0.25834751,
        0.2523894 , 0.26028726, 0.19479325, 0.1873401 , 0.20989066,
        0.2553898 , 0.21248396, 0.26848375, 0.26534232, 0.2787429 ,
        0.26746896, 0.3139312 , 0.21933157, 0.21344325, 0.33650128,
        0.29435102, 0.25292239, 0.32886934, 0.31971647, 0.29192191,
        0.30989502, 0.25844693, 0.2832191 , 0.23981708, 0.24776722,
        0.23528986, 0.24054324, 0.3472929 , 0.34032116, 0.30355299,
        0.35143287, 0.32365001, 0.29560084, 0.31473719, 0.26980099,
        0.30729119, 0.25581414, 0.32521638, 0.350383  , 0.2789121 ,
        0.1877